<a href="https://colab.research.google.com/github/dineshsharma143/AI-Invoice-Automation-week-4-Assignment/blob/main/Neural_Network_Classification_on_AI_Invoice_Automation_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam

In [ ]:
df = pd.read_csv("/content/AI Invoice Automation Dataset.csv")

df.head()

In [ ]:
df = df.drop(
    columns=[
        "Invoice_ID",
        "Invoice_Number",
        "Invoice_Date",
        "Due_Date"
    ],
    errors="ignore"
)

In [ ]:
X = df.drop(
    "Status",
    axis=1
)

y = df["Status"]

In [ ]:
encoder = LabelEncoder()

for col in X.select_dtypes(include="object").columns:
    X[col] = encoder.fit_transform(X[col])

In [ ]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

print(target_encoder.classes_)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)

In [ ]:
dnn_model = Sequential()


dnn_model.add(
    Dense(
        128,
        activation="relu",
        input_shape=(X_train.shape[1],)
    )
)


dnn_model.add(
    Dense(
        64,
        activation="relu"
    )
)


dnn_model.add(
    Dropout(
        0.3
    )
)


dnn_model.add(
    Dense(
        32,
        activation="relu"
    )
)


dnn_model.add(
    Dense(
        3,
        activation="softmax"
    )
)


dnn_model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


dnn_model.summary()

In [ ]:
dnn_history = dnn_model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

In [ ]:
dnn_probability = dnn_model.predict(
    X_test
)


dnn_prediction = np.argmax(
    dnn_probability,
    axis=1
)

In [ ]:
X_train_cnn = np.expand_dims(
    X_train,
    axis=2
)


X_test_cnn = np.expand_dims(
    X_test,
    axis=2
)


print(X_train_cnn.shape)

In [ ]:
cnn_model = Sequential()


cnn_model.add(
    Conv1D(
        filters=64,
        kernel_size=2,
        activation="relu",
        input_shape=(
            X_train_cnn.shape[1],
            1
        )
    )
)


cnn_model.add(
    MaxPooling1D(
        pool_size=2
    )
)


cnn_model.add(
    Flatten()
)


cnn_model.add(
    Dense(
        64,
        activation="relu"
    )
)


cnn_model.add(
    Dropout(
        0.3
    )
)


cnn_model.add(
    Dense(
        3,
        activation="softmax"
    )
)



cnn_model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


cnn_model.summary()

In [ ]:
cnn_history = cnn_model.fit(
    X_train_cnn,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

In [ ]:
cnn_probability = cnn_model.predict(
    X_test_cnn
)


cnn_prediction = np.argmax(
    cnn_probability,
    axis=1
)

In [ ]:
dnn_accuracy = accuracy_score(
    y_test,
    dnn_prediction
)


cnn_accuracy = accuracy_score(
    y_test,
    cnn_prediction
)



comparison = pd.DataFrame(
    {
        "Model":[
            "DNN",
            "CNN"
        ],

        "Accuracy":[
            dnn_accuracy,
            cnn_accuracy
        ]
    }
)


comparison

In [ ]:
print(
    classification_report(
        y_test,
        dnn_prediction,
        target_names=target_encoder.classes_
    )
)

In [ ]:
print(
    classification_report(
        y_test,
        cnn_prediction,
        target_names=target_encoder.classes_
    )
)

In [ ]:
precision = precision_score(
    y_test,
    cnn_prediction,
    average="weighted"
)


recall = recall_score(
    y_test,
    cnn_prediction,
    average="weighted"
)


f1 = f1_score(
    y_test,
    cnn_prediction,
    average="weighted"
)


print("Precision:", precision)

print("Recall:", recall)

print("F1 Score:", f1)

In [ ]:
cm = confusion_matrix(
    y_test,
    cnn_prediction
)


display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_encoder.classes_
)


display.plot()

plt.title(
    "CNN Confusion Matrix"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))


plt.plot(
    dnn_history.history["val_accuracy"],
    label="DNN Validation Accuracy"
)


plt.plot(
    cnn_history.history["val_accuracy"],
    label="CNN Validation Accuracy"
)


plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title(
    "DNN vs CNN Performance"
)


plt.legend()

plt.show()